# SML Project 1 — Obstacle Distance Estimation
**Pipeline:** StandardScaler → PCA → HistGradientBoostingRegressor (+ optional stacking)

Grading thresholds: MAE ≤ 28 cm (1.0) | ≤ 18 cm (4.0) | ≤ 8 cm (6.0)

## 0. Environment Setup

In [ ]:
import sys, os

# ── Auto-detect environment ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/project1
    print("Running on Google Colab")
else:
    # Change this to your local project path if needed
    PROJECT_DIR = r"C:/Users/aarya/OneDrive/Desktop/Distance-Recognition-SML1-"
    os.chdir(PROJECT_DIR)
    print(f"Running locally from: {os.getcwd()}")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

print("All imports OK")

## 2. Load Config & Data

In [ ]:
# Edit config.yaml before running:
#   load_rgb: True
#   downsample_factor: 5

config = load_config()
images, distances = load_dataset(config, split="train")

print(f"Images shape : {images.shape}")
print(f"Distances shape: {distances.shape}")
print(f"Distance range : {distances.min():.2f} m – {distances.max():.2f} m")

## 3. Quick Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
axes[0].hist(distances * 100, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Distance (cm)')
axes[0].set_ylabel('Count')
axes[0].set_title('Label Distribution')

# Mean pixel intensity vs distance
mean_intensity = images.mean(axis=1)
axes[1].scatter(distances * 100, mean_intensity, alpha=0.3, s=5, color='coral')
axes[1].set_xlabel('Distance (cm)')
axes[1].set_ylabel('Mean Pixel Intensity')
axes[1].set_title('Pixel Intensity vs Distance')

plt.tight_layout()
plt.show()

## 4. Train / Validation / Test Split
70 % train | 15 % validation | 15 % test

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, distances, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train : {X_train.shape[0]} samples")
print(f"Val   : {X_val.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")

## 5. Preprocessing — Scaler + PCA
**Rule:** fit only on training data, then transform val/test.

In [ ]:
N_COMPONENTS = 100  # tune this — try 50, 100, 150, 200

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

explained = pca.explained_variance_ratio_.cumsum()[-1] * 100
print(f"PCA with {N_COMPONENTS} components explains {explained:.1f}% of variance")
print(f"Reduced feature shape: {X_train_pca.shape}")

In [ ]:
# Plot cumulative explained variance to help choose n_components
pca_full = PCA(n_components=min(300, X_train_scaled.shape[1]), random_state=42)
pca_full.fit(X_train_scaled)

cum_var = pca_full.explained_variance_ratio_.cumsum()
plt.figure(figsize=(8, 4))
plt.plot(cum_var * 100)
plt.axhline(95, color='red', linestyle='--', label='95% threshold')
plt.axvline(N_COMPONENTS, color='green', linestyle='--', label=f'n={N_COMPONENTS}')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance (%)')
plt.title('PCA Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Baseline — Ridge Regression

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_pca, y_train)

val_pred_ridge = ridge.predict(X_val_pca)
mae_ridge = mean_absolute_error(y_val, val_pred_ridge) * 100
print(f"Ridge  — Val MAE: {mae_ridge:.2f} cm")

## 7. Main Model — HistGradientBoostingRegressor

In [ ]:
hgbr = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
)
hgbr.fit(X_train_pca, y_train)

val_pred_hgbr = hgbr.predict(X_val_pca)
mae_hgbr = mean_absolute_error(y_val, val_pred_hgbr) * 100
r2_hgbr  = r2_score(y_val, val_pred_hgbr) * 100
print(f"HGBR  — Val MAE: {mae_hgbr:.2f} cm  |  R²: {r2_hgbr:.1f}%")

## 8. Hyperparameter Tuning — GridSearchCV
⚠️ This cell can take a while. Reduce the param grid for faster iteration.

In [ ]:
param_grid = {
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth':     [4, 6, 8],
    'max_iter':      [200, 400],
}

grid_search = GridSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_pca, y_train)

print("Best params:", grid_search.best_params_)
best_hgbr = grid_search.best_estimator_

val_pred_best = best_hgbr.predict(X_val_pca)
mae_best = mean_absolute_error(y_val, val_pred_best) * 100
print(f"Tuned HGBR — Val MAE: {mae_best:.2f} cm")

## 9. Optional — Stacking (HGBR + RandomForest → Ridge)

In [ ]:
# Uncomment to try stacking

# estimators = [
#     ('hgbr', HistGradientBoostingRegressor(
#         **grid_search.best_params_, random_state=42)),
#     ('rf',   RandomForestRegressor(
#         n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)),
# ]
# stack = StackingRegressor(
#     estimators=estimators,
#     final_estimator=Ridge(),
#     cv=5,
#     n_jobs=-1,
# )
# stack.fit(X_train_pca, y_train)
# 
# val_pred_stack = stack.predict(X_val_pca)
# mae_stack = mean_absolute_error(y_val, val_pred_stack) * 100
# print(f"Stacking — Val MAE: {mae_stack:.2f} cm")

## 10. Model Comparison Summary

In [ ]:
results = {
    'Ridge (baseline)': mae_ridge,
    'HGBR (default)':   mae_hgbr,
    'HGBR (tuned)':     mae_best,
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(results.keys()), list(results.values()), color=['#aac4e0', '#6baed6', '#2171b5'])
ax.axvline(28, color='orange', linestyle='--', label='Grade 1.0 (28 cm)')
ax.axvline(18, color='gold',   linestyle='--', label='Grade 4.0 (18 cm)')
ax.axvline(8,  color='green',  linestyle='--', label='Grade 6.0 (8 cm)')
ax.set_xlabel('Val MAE (cm)')
ax.set_title('Model Comparison on Validation Set')
ax.legend()
for bar, val in zip(bars, results.values()):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f} cm', va='center')
plt.tight_layout()
plt.show()

## 11. Final Evaluation on Held-Out Test Set
⚠️ Run this only once, after all tuning is complete.

In [ ]:
# Swap best_hgbr for stack if stacking was better
final_model = best_hgbr

test_pred = final_model.predict(X_test_pca)
print("=== FINAL TEST SET RESULTS ===")
print_results(y_test, test_pred)

## 12. Generate Kaggle Submission
Retrain final model on ALL labelled data (train+val+test), then predict on Kaggle test images.

In [ ]:
# ── Retrain on full labelled dataset ─────────────────────────────────────────
X_all_scaled = scaler.fit_transform(images)         # refit scaler on all data
X_all_pca    = pca.fit_transform(X_all_scaled)      # refit PCA on all data

final_model.set_params(**grid_search.best_params_)  # keep best hyperparams
final_model.fit(X_all_pca, distances)
print("Final model retrained on full dataset")

# ── Load and transform Kaggle test images ────────────────────────────────────
kaggle_images = load_test_dataset(config)
kaggle_images = np.array(kaggle_images)

kaggle_scaled = scaler.transform(kaggle_images)
kaggle_pca    = pca.transform(kaggle_scaled)

kaggle_pred = final_model.predict(kaggle_pca)

save_results(kaggle_pred)
print(f"Saved prediction.csv with {len(kaggle_pred)} predictions")
print(f"Predicted distance range: {kaggle_pred.min():.2f} m – {kaggle_pred.max():.2f} m")

---
### Config Quick-Reference
| `downsample_factor` | `load_rgb` | Features (before PCA) | Notes |
|---|---|---|---|
| 10 | False | 900 | Fast iteration / baseline |
| 5  | False | 3 600 | Good balance |
| 5  | True  | 10 800 | Recommended starting point |
| 3  | True  | 30 000 | High quality, needs more RAM |
| 1  | False | 90 000 | Max grayscale — may be slow |